### Import Dependencies

In [8]:
from google import genai
from google.genai import types
import os
from qdrant_client import QdrantClient
gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

### Embedding function

In [9]:
### Other task_type = SEMANTIC_SIMILARITY, CLASSIFICATION, CLUSTERING, RETRIEVAL_DOCUMENT, CODE_RETRIEVAL_QUERY, QUESTION_ANSWERING, FACT_VERIFICATION

def get_embeddings(text, task_type="SEMANTIC_SIMILARITY", model="gemini-embedding-001"):
    result = gemini_client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return result.embeddings[0].values

### Retrieval Function

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [5]:
def retrieve_data(query, k=5):
    query_embedding = get_embeddings(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )
    retrieved_context_ids=[]
    retrieved_context=[]
    similarity_scores=[]
    retrieved_context_ratings=[]

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [10]:
retrieved_context = retrieve_data("do you have any smart watches?", k=10)

In [11]:
retrieved_context

{'retrieved_context_ids': ['B09SNWJC5N',
  'B0BCF3KRMM',
  'B0B5HN5RNC',
  'B0C1CL9Y8T',
  'B09V1F24X4',
  'B09QQ2J8XN',
  'B09YHBXZC8',
  'B0C6PKB5QW',
  'B0BC4PGXFK',
  'B08DLWKXGL'],
 'retrieved_context': ["Smart Watch, Tykoit Smartwatch 41mm with Heart Rate Monitor & Sleep Tracking & SpO2, IP68 Waterproof Fitness Tracker with Step/Calorie Counter Compatible with Android and iOS Phones Heart Rate, Sleep and SpO2 Monitor: This smart watch built in high performance motion sensor automatically detects your heart rate in real time, and also measure your SpO2 and deep, light sleep state which can help you better understand your health. (NOTE: the data cannot be used as a medical-grade test) All-day Activity Tracking: This smartwatch has 8 kinds of sports modes (walking, running, treadmill, bicycle, yoga and other workouts). The fitness tracker display shows your accurate steps, distance, calories, active minutes and you can also see more detailed data on App. Customizable Watch Face: Our

In [12]:
### Format retrieved context function

def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context

In [13]:
preprocessed_context = process_context(retrieved_context)

In [15]:
print(preprocessed_context)

- ID: B09SNWJC5N, rating: 4.1, description: Smart Watch, Tykoit Smartwatch 41mm with Heart Rate Monitor & Sleep Tracking & SpO2, IP68 Waterproof Fitness Tracker with Step/Calorie Counter Compatible with Android and iOS Phones Heart Rate, Sleep and SpO2 Monitor: This smart watch built in high performance motion sensor automatically detects your heart rate in real time, and also measure your SpO2 and deep, light sleep state which can help you better understand your health. (NOTE: the data cannot be used as a medical-grade test) All-day Activity Tracking: This smartwatch has 8 kinds of sports modes (walking, running, treadmill, bicycle, yoga and other workouts). The fitness tracker display shows your accurate steps, distance, calories, active minutes and you can also see more detailed data on App. Customizable Watch Face: Our smartwatches support customizable watch faces with your favorite pictures to suit your daily mood and occasion. And the fitness watch with 1.4 inch color full touchs

### Create prompt template function

In [16]:
def build_prompt(query, preprocessed_context):
    prompt = f"""
    You are a helpful assistant that can answer questions about the products in stock.
    You are given a question and a list of context.
    You need to answer the question based on the product descriptions.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use the word "context" in your answer, refer it as the available products.
    - Do not use markdown formatting.

    Context:
    {preprocessed_context}

    Question: 
    {query}
    """
    return prompt

In [17]:
prompt = build_prompt("do you have any smart watches?", preprocessed_context)

In [18]:
print(prompt)


    You are a helpful assistant that can answer questions about the products in stock.
    You are given a question and a list of context.
    You need to answer the question based on the product descriptions.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use the word "context" in your answer, refer it as the available products.
    - Do not use markdown formatting.

    Context:
    - ID: B09SNWJC5N, rating: 4.1, description: Smart Watch, Tykoit Smartwatch 41mm with Heart Rate Monitor & Sleep Tracking & SpO2, IP68 Waterproof Fitness Tracker with Step/Calorie Counter Compatible with Android and iOS Phones Heart Rate, Sleep and SpO2 Monitor: This smart watch built in high performance motion sensor automatically detects your heart rate in real time, and also measure your SpO2 and deep, light sleep state which can help you better understand your health. (NOTE: the data cannot be used as a medical-grade test) All-day Activity Tracking: This s

### Generate Answer

In [19]:
def generate_answer(prompt):
    response = gemini_client.models.generate_content(
    contents=[prompt],
    model="gemini-2.5-flash",
)
    return response.text

In [20]:
print(generate_answer(prompt))

Yes, we have several smart watches available:

- Smart Watch, Tykoit Smartwatch 41mm with Heart Rate Monitor & Sleep Tracking & SpO2, IP68 Waterproof Fitness Tracker with Step/Calorie Counter Compatible with Android and iOS Phones.
- Military Smart Watch Men (Answer/Make Calls), 2023 Newest Outdoor Bluetooth Smart Watch for Android iPhone, 5 ATM Waterproof Tactical Watch, Rugged Fitness Tracker with Heart Rate/SpO2/AI Voice, Silver.
- PlayBetter Garmin Instinct 2 Solar Tactical (Black) Rugged GPS Smartwatch | Power Bundle TPU Screen Protectors & Portable Charger | 2022 Men's Military Watch with Tactical Features | Large, 45mm.


### Combine RAG Pipeline

In [21]:
def rag_pipeline(query, topk_k=5):
    retrieved_context = retrieve_data(query, topk_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(query, preprocessed_context)
    answer = generate_answer(prompt)
    return answer

In [22]:
print(rag_pipeline("do you have any smart watches?"))

Yes, we have several smartwatches available:

*   A Tykoit Smartwatch 41mm with a heart rate monitor, sleep tracking, and SpO2, which is IP68 waterproof and compatible with Android and iOS phones. It features 8 sports modes, customizable watch faces, and smart notifications.
*   A Military Smart Watch for Men that allows answering/making calls, is 5 ATM waterproof, and includes heart rate, SpO2, and AI voice features. It's customized for adventurers with military-grade tests and 20 sports modes.
*   A PlayBetter Garmin Instinct 2 Solar Tactical (Black) Rugged GPS Smartwatch. This model offers unlimited battery life in smartwatch mode with solar charging, dedicated tactical features like stealth mode, and all-day health monitoring including VO2 Max, heart rate, sleep, and blood oxygen level. It also comes with a power bundle including screen protectors and a portable charger.


In [23]:
print(rag_pipeline("could you suggest me some headphones? preferably with a rating of over 4 stars"))

Certainly! Based on the available products, here are some headphones with a rating of over 4 stars:

- ID: B09V1F24X4, rating: 4.2, description: Wireless Earbuds with 4-Mic and Wireless Charging Case Waterproof 50H Playback Bluetooth Headphones LED Power Display Stereo Earphones, Touch Control in-Ear Headset with USB-C for Sports Work Black
- ID: B0BD8FXHPB, rating: 4.4, description: WeurGhy Wireless Earbuds, Bluetooth 5.3 Headphones, Bluetooth Earbuds with Stereo Sound, 42 Hrs of Playtime, IPX7 Waterproof, LED Display, Over Ear Earphones with Mic for Sports Running Workout Gym
- ID: B0BNNTDHNX, rating: 4.4, description: VuyKoo Bluetooth Headphones with Microphone/RGB LED Light Up, Cat Ear Wireless Headphones, Stereo Gaming Headset for Cellphone/PC/Laptop/Tablet/TV Kids Girls & Boys Teens/Birthday Gift (Purple)
- ID: B09XKJFXB4, rating: 4.4, description: Rhystereo Active Noise Cancelling Headphones, Black Wireless Headphones w/Mics, 45H Playtime USB-C Fast Charge, Deep Bass, BT 5.0, Co